# Create Work Citations (oxjob #2314, shadow)

One row per citation: the citing work, the cited work, and every source that holds it (each filed reference list with its
version tier, or legacy MAG). Built once per night after `parse_referenced_works` and `Merged_Work_Ids_Map`.

**Shadow:** nothing reads `openalex.works.work_citations` yet. `Works_Enriched` still serves the union of our lists, legacy
MAG and the frozen per-location arrays; an error here only logs. The switch (job PLAN step 4) makes `Works_Enriched` set
each work's `referenced_works` from the rows with `served = true`.

`served` is the first version's rule (Casey, 16 Nov 2026): every citation the work's best version's lists link (rule B:
published and accepted, then submitted, then other) plus every legacy MAG citation. `in_best_version` and
`from_legacy_mag` keep the parts, so a rule change needs no rebuild. On 16 Nov's tables: 3,578,230,346 rows,
3,566,614,633 served (today's arrays: 3,593,082,030), 39 GB, 3.5 min on shared-2xl.

In [0]:
-- oxjob #2314, SHADOW: one row per citation (citing work, cited work) that any current input holds, with every source that
-- holds it, built once per night. Nothing reads this table yet; any error only logs, so End 2 End goes on.
-- Inputs: our filed lists (work_references rows of the lists in work_reference_lists_current, each with its version tier:
-- published and accepted 1, submitted 2, other 3) and legacy MAG (openalex.mid.citation). The frozen per-location arrays
-- (location_enrichments, via openalex_works_base) are not read. Both ends follow tonight's merged_work_ids_map; no work
-- cites itself; the citing work is one of tonight's works.
-- served = the first version's rule (Casey 16 Nov): the best version's lists (rule B, Casey 6 and 9 Nov) plus every
-- legacy MAG citation. The MAG rule (MAG only where the best version's lists link nothing) waits for Casey's go.
BEGIN
  DECLARE EXIT HANDLER FOR SQLEXCEPTION
  BEGIN
    DECLARE msg STRING;
    GET DIAGNOSTICS CONDITION 1 msg = MESSAGE_TEXT;
    SELECT concat('work_citations (shadow, oxjob #2314): build failed, nothing reads it yet so End 2 End goes on: ', msg) AS work_citations;
  END;
  CREATE OR REPLACE TABLE openalex.works.work_citations
  CLUSTER BY (citing_work_id)
  TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '7 days', 'delta.logRetentionDuration' = '7 days')
  AS
  WITH m AS (SELECT id AS k, merge_into_id AS v FROM openalex.works.merged_work_ids_map),
  lists AS (
    SELECT l.native_id, l.native_id_namespace, l.provenance, coalesce(mc.v, l.citing_work_id) AS citing_work_id,
      CASE WHEN l.version_rank <= 2 THEN 1 WHEN l.version_rank = 3 THEN 2 ELSE 3 END AS version_tier
    FROM openalex.works.work_reference_lists_current l
    LEFT JOIN m mc ON mc.k = l.citing_work_id
  ),
  best AS (SELECT citing_work_id, min(version_tier) AS best_tier FROM lists GROUP BY citing_work_id),
  edges AS (
    SELECT coalesce(mc.v, wr.citing_work_id) AS citing_work_id, coalesce(md.v, wr.cited_work_id) AS cited_work_id,
      named_struct('provenance', wr.provenance, 'native_id_namespace', wr.native_id_namespace, 'native_id', wr.native_id,
                   'version_tier', l.version_tier) AS source
    FROM openalex.works.work_references wr
    LEFT JOIN lists l
      ON l.native_id = wr.native_id AND l.native_id_namespace = wr.native_id_namespace AND l.provenance = wr.provenance
    LEFT JOIN m mc ON mc.k = wr.citing_work_id
    LEFT JOIN m md ON md.k = wr.cited_work_id
    WHERE wr.cited_work_id IS NOT NULL
    UNION ALL
    SELECT coalesce(mc.v, c.paper_id), coalesce(md.v, c.paper_reference_id),
      named_struct('provenance', 'legacy-mag', 'native_id_namespace', CAST(NULL AS STRING), 'native_id', CAST(NULL AS STRING),
                   'version_tier', CAST(NULL AS INT))
    FROM openalex.mid.citation c
    LEFT JOIN m mc ON mc.k = c.paper_id
    LEFT JOIN m md ON md.k = c.paper_reference_id
  ),
  pairs AS (
    SELECT citing_work_id, cited_work_id, array_sort(collect_set(source)) AS sources
    FROM edges
    WHERE citing_work_id IS NOT NULL AND cited_work_id IS NOT NULL AND citing_work_id <> cited_work_id
    GROUP BY citing_work_id, cited_work_id
  )
  SELECT p.citing_work_id, p.cited_work_id, p.sources,
    exists(p.sources, s -> s.provenance <> 'legacy-mag' AND s.version_tier = b.best_tier) AS in_best_version,
    exists(p.sources, s -> s.provenance = 'legacy-mag') AS from_legacy_mag,
    exists(p.sources, s -> s.provenance <> 'legacy-mag' AND s.version_tier = b.best_tier)
      OR exists(p.sources, s -> s.provenance = 'legacy-mag') AS served,
    current_timestamp() AS built_at
  FROM pairs p
  LEFT JOIN best b ON b.citing_work_id = p.citing_work_id
  LEFT SEMI JOIN openalex.works.openalex_works_base w ON w.id = p.citing_work_id;
END
